# Prepare Table 013

Download and clean this ABS source, save its workbook and CSV, then publish its Delta table directly. Each notebook runs independently.

Use the job parameters described in [README](../README.md), and fill in `catalog` and `schema` in the final publishing cell.

Table 013 covers labour force status for people aged 15–24 in Australia, by sex. The descriptions give measure before sex.

The reference is the [July 2026 workbook](https://www.abs.gov.au/statistics/labour/employment-and-unemployment/labour-force-australia/jul-2026/62020013.xlsx); the first run selects August 2026.


## 1. Setup and source settings

Import the tools used by the notebook and set the dataset names, filenames, download address and workbook layout. Edit these settings when ABS changes this source, and review the workbook before updating the reference file.

In [ ]:
%pip install openpyxl

In [ ]:
# Import the tools for dates, file paths, tables, downloads and notebook displays.
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import requests
from IPython.display import display

# Set the dataset name, filenames and ABS download address.
dataset = "Table 013"
dataset_key = "table_013"
baseline_file = "Table 013.xlsx"
download_file = "62020013.xlsx"
base_url = "https://www.abs.gov.au/statistics/labour/employment-and-unemployment/labour-force-australia/"

# List the sheet containing monthly data.
data_sheets = ["Data1"]

# Name the ten rows that describe each series, in their workbook order.
metadata_fields = ["description", "unit", "series_type", "data_type", "frequency", "collection_month", "series_start", "series_end", "n_obs", "series_id"]

## 2. Job parameters and next release

The job supplies `run_date` (`{{job.start_time.iso_date}}`, the job start date in UTC), `job_run_id` (`{{job.run_id}}`) and `workspace_path` (the absolute `/Workspace/...` path to the project folder). Table 013 determines its release month automatically; it does not use the `release_month` job parameter.

Keep the reference workbook in `referencedatasets/`. The next release is one month after the later of its latest Data1 month and the latest completed Table 013 release recorded in `outputs/table_013/*/sources.csv`. A run is completed only after the CSV and Delta table have been written. Install pandas, openpyxl and requests in the task environment.

In [ ]:
# Create text inputs for the run date, job run ID and workspace project path.
dbutils.widgets.text("run_date", "", "Job run date (YYYY-MM-DD, UTC)")
dbutils.widgets.text("job_run_id", "", "Job run ID")
dbutils.widgets.text("workspace_path", "", "Workspace project path")

# Read the input values and convert the workspace path into a file path object.
run_date = dbutils.widgets.get("run_date")
job_run_id = dbutils.widgets.get("job_run_id")
workspace = Path(dbutils.widgets.get("workspace_path"))
# Check that the job run ID is filled in and is not an unresolved {{...}} placeholder.
assert job_run_id and "{{" not in job_run_id, "Supply the resolved job run ID"
# Check that the workspace project path is filled in.
assert dbutils.widgets.get("workspace_path"), "Supply workspace_path"

# Set the reference folder and the output folder named with the run date and job run ID.
baseline = workspace / "referencedatasets"
output = workspace / "outputs" / dataset_key
run = output / f"{run_date}_{job_run_id}"

# Read the last month recorded in the reference workbook’s Data1 sheet.
baseline_latest_month = pd.to_datetime(pd.read_excel(baseline / baseline_file, sheet_name="Data1", header=None).iloc[-1, 0])
# Find the source records from completed Table 013 runs and read their release months.
completed_sources = output.glob("*/sources.csv")
processed_months = [pd.to_datetime(pd.read_csv(source_file)["reference_period"]).max() for source_file in completed_sources]

# Choose the next month after the latest reference or completed release, and format it as YYYY-MM.
download_month = max([baseline_latest_month, *processed_months]) + pd.offsets.MonthBegin(1)
release_month = download_month.strftime("%Y-%m")
# Display the dataset, selected release month and output folder.
print(dataset, release_month, run)

## 3. Download the release workbook

In [ ]:
# Build the download URL from the ABS address, selected release month and filename.
source_url = base_url + f"{download_month:%b-%Y}/".lower() + download_file
# Record the current time in UTC for the source record.
retrieved_at = datetime.now(timezone.utc).isoformat()
# Download the workbook and stop if the server reports an unsuccessful response.
response = requests.get(source_url, timeout=120, headers={"Cache-Control": "no-cache"})
response.raise_for_status()
# Check that the response starts with the ZIP file marker used by XLSX files.
assert response.content.startswith(b"PK"), f"{dataset}: download is not an XLSX file"

# Create the raw-data folder and save the downloaded workbook there.
(run / "raw").mkdir(parents=True, exist_ok=True)
(run / "raw" / download_file).write_bytes(response.content)

# Display the download URL.
print(source_url)

## 4. Read and validate the workbook

Read the reference and downloaded workbooks, then compare their sheets, header labels, series IDs and series definitions. Check the latest month in each data sheet against the selected release month.

In [ ]:
# Read every sheet from the reference and downloaded workbooks, keeping the first row as data.
table_013_base = pd.read_excel(baseline / baseline_file, sheet_name=None, header=None)
table_013_book = pd.read_excel(run / "raw" / download_file, sheet_name=None, header=None)

# Check that the downloaded workbook has the same sheet names as the reference.
dataset = "Table 013"
assert set(table_013_book) == set(table_013_base), f"{dataset}: workbook sheets changed"
# Locate the first monthly data row and the row containing series IDs.
first_data_row = len(metadata_fields)
series_id_row = metadata_fields.index("series_id")
# Select the series details to compare, excluding end dates and observation counts.
definition_fields = ["description", "unit", "series_type", "data_type", "frequency", "collection_month", "series_start"]
definition_rows = [metadata_fields.index(field) for field in definition_fields]

# Create an empty list to collect a summary of each sheet.
sheet_checks = []

# Check the named data sheet against its reference sheet.
for sheet in data_sheets:
    raw_sheet = table_013_book[sheet]
    baseline_sheet = table_013_base[sheet]
    # Check that the first ten row labels match those in the reference sheet.
    pd.testing.assert_series_equal(raw_sheet.iloc[:first_data_row, 0], baseline_sheet.iloc[:first_data_row, 0], obj=f"{dataset}/{sheet}: header rows")
    # Read the series IDs and check that none are missing or repeated within this sheet.
    series_ids = raw_sheet.iloc[series_id_row, 1:]
    assert series_ids.notna().all() and series_ids.is_unique, f"{dataset}/{sheet}: missing or duplicate series IDs"

    # Select the definition rows from each sheet and line up their columns by series ID.
    sheet_definitions = raw_sheet.iloc[definition_rows, 1:].set_axis(series_ids, axis=1).sort_index(axis=1)
    baseline_definitions = baseline_sheet.iloc[definition_rows, 1:].set_axis(baseline_sheet.iloc[series_id_row, 1:], axis=1).sort_index(axis=1)
    # Compare the selected series details with the reference and stop if they differ.
    pd.testing.assert_frame_equal(sheet_definitions, baseline_definitions, obj=f"{dataset}/{sheet}: series definitions")

    # Read the dates from the first column and check that the latest matches the selected release month.
    dates = pd.to_datetime(raw_sheet.iloc[first_data_row:, 0])
    assert dates.max() == download_month, f"{dataset}/{sheet}: latest month does not match {release_month}"
    # Record the number of months, series and observation cells, plus the latest month.
    sheet_checks.append((dataset, sheet, dates.nunique(), len(series_ids), len(dates) * len(series_ids), dates.max()))

# Display a summary of the records checked in each data sheet.
display(pd.DataFrame(sheet_checks, columns=["dataset", "sheet", "months", "series", "observations", "latest_month"]))

## 5. Reshape and clean the workbook

Reshape the Data1 sheet's monthly values into rows and add the measure, sex, age, region and unit labels. Check the resulting labels and row counts as part of the cleaning steps.

In [ ]:
# Create empty lists to collect series details and monthly values from each sheet.
metadata_frames, observation_frames = [], []

# Reshape the data sheet that passed the workbook checks.
for sheet in data_sheets:
    raw_sheet = table_013_book[sheet]
    series_ids = raw_sheet.iloc[series_id_row, 1:]
    # Turn the series details into one row per series, label the columns and sort by ID.
    sheet_metadata = raw_sheet.iloc[:first_data_row, 1:].T.set_axis(metadata_fields, axis=1).set_index("series_id").sort_index()
    # Reshape the monthly values into date, series ID and value columns, and convert the dates.
    sheet_observations = (
        raw_sheet.iloc[first_data_row:].set_axis(["date", *series_ids], axis=1).melt("date", var_name="series_id").astype({"date": "datetime64[ns]"})
    )
    # Convert the observation values to numbers.
    sheet_observations["value"] = pd.to_numeric(sheet_observations["value"])
    # Collect this sheet’s values and series details, adding their dataset and sheet labels.
    observation_frames.append(sheet_observations.assign(dataset=dataset))
    metadata_frames.append(sheet_metadata.reset_index().assign(dataset=dataset, sheet=sheet))

# Combine the results from all data sheets into two tables.
table_013_metadata = pd.concat(metadata_frames, ignore_index=True)
table_013 = pd.concat(observation_frames, ignore_index=True)

# Remove surrounding spaces and trailing semicolons, then split each description into columns.
description_parts = table_013_metadata["description"].str.strip().str.rstrip(";").str.split(";", expand=True)
# Remove spaces and leading > markers from each part of the description.
description_parts = description_parts.apply(lambda s: s.str.strip().str.lstrip(">").str.strip())
# Copy the first two description parts into the measure and sex columns.
table_013_metadata["measure"] = description_parts[0]
table_013_metadata["sex"] = description_parts[1]
# Label every series as covering people aged 15–24 in Australia.
table_013_metadata["region"] = "Australia"
table_013_metadata["age"] = "15-24 years"
# Add an apostrophe to the thousands unit label; leave other unit labels unchanged.
table_013_metadata["unit"] = table_013_metadata["unit"].replace("000", "'000")
# Check that every series has a measure, sex, region and age label.
assert table_013_metadata[["measure", "sex", "region", "age"]].notna().all().all()

# Count the rows for each dataset, date and series ID combination before joining.
observation_keys = ["dataset", "date", "series_id"]
keys_before_join = table_013[observation_keys].value_counts(sort=False).sort_index()

# Attach the series labels and units, checking that each dataset and series ID has only one matching row of details.
table_013 = table_013.merge(
    table_013_metadata[["dataset", "series_id", "measure", "sex", "age", "region", "series_type", "data_type", "unit"]],
    on=["dataset", "series_id"],
    validate="many_to_one",
)
# Check that every dataset, date and series ID combination still has the same row count.
pd.testing.assert_series_equal(
    keys_before_join, table_013[observation_keys].value_counts(sort=False).sort_index(), obj="Table 013 metadata join: observation keys and counts"
)
# Display the table size and the first five rows for review.
print(table_013.shape)
display(table_013[["date", "series_id", "measure", "region", "sex", "age", "series_type", "data_type", "unit", "value"]].head())

## 6. Save the cleaned CSV

Save the cleaned data under `outputs/table_013/<run_date>_<job_run_id>/`. Runs without `sources.csv` are incomplete and do not advance the next release. Retrying an incomplete run with the same date and job run ID overwrites its raw workbook and CSV.

In [ ]:
# Copy the columns to include in the output, in their required order.
df = table_013[["date", "series_id", "measure", "sex", "age", "region", "series_type", "data_type", "unit", "value"]].copy()

# Name the CSV using the dataset and release month, then save it without row numbers and with dates as YYYY-MM-DD.
csv_file = run / f"{dataset_key}_{release_month}.csv"
df.to_csv(csv_file, index=False, date_format="%Y-%m-%d")
# Display the table size and the first five rows for review.
print(df.shape)
display(df.head())

## 7. Publish the Delta table

Replace `YOUR_CATALOG` and `YOUR_SCHEMA` with your existing destination names. `dataset_key` supplies this source's table name. Run this cell in Databricks after the CSV export.

The write creates a missing Delta table or overwrites its data. It uses the cleaned `df` already in memory; no staging Delta table or separate publisher is needed. This source updates independently of the other sources. Plain overwrite does not automatically replace an existing table's schema.

After publishing, write `sources.csv` with this release's source details. Its presence tells the next run that this release completed successfully. A full rerun after completion selects the next release; retry only the publishing cell to republish the current in-memory data.

In [ ]:
# Enter the destination catalogue and schema, then build the full table name.
catalog = "YOUR_CATALOG"
schema = "YOUR_SCHEMA"
table = f"{catalog}.{schema}.{dataset_key}"

# Convert the cleaned data to a Spark table and store dates without a time component.
sdf = spark.createDataFrame(df)
sdf = sdf.withColumn("date", sdf["date"].cast("date"))

# Create the Delta table or overwrite its existing data.
sdf.write.format("delta").mode("overwrite").saveAsTable(table)

# Collect the dataset, filenames, download URL, release month and retrieval time into one source record.
sources = pd.DataFrame(
    [(dataset, baseline_file, download_file, source_url, download_month, retrieved_at, csv_file.name)],
    columns=["dataset", "baseline_file", "download_file", "source_url", "reference_period", "retrieved_at_utc", "output_file"],
)
# Save sources.csv after publication to mark this release as completed for the next run.
sources.to_csv(run / "sources.csv", index=False, date_format="%Y-%m-%d")

# Display the published row count, destination table and source record.
print(f"Published {len(df):,} rows to {table}")
display(sources)